<a href="https://colab.research.google.com/github/adzetto/marine_analysis/blob/main/su-seviyesi/colab_baslat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bozyazı deniz seviyesi analizi — zinciri çalıştır

Bu defter **hesabı yapar**. Sonuçları anlatımıyla, formülleriyle ve
grafikleriyle görmek için `analiz_raporu.ipynb` defterini açın.

**Çalışma zamanını yüksek RAM'e alın.**  
`Çalışma zamanı → Çalışma zamanı türünü değiştir → Yüksek RAM`

Ağır adım yalnızca `05`: 15 dakikalık çözünürlükte 300.000 nokta için
UTide'ın tasarım matrisi birkaç GB istiyor. `05` çözümü diske yazdığı
için `06` ve `07` onu tekrar hesaplamaz — hepsi hafif.

## 1. Kurulum

Tekrar tekrar çalıştırılabilir; her seferinde temiz klon alır.

In [ ]:
%cd /content
!rm -rf /content/marine_analysis
!git clone -q https://github.com/adzetto/marine_analysis.git
%cd /content/marine_analysis/su-seviyesi
!pip install -q -r requirements.txt

import psutil, os
gb = psutil.virtual_memory().total / 1e9
print(f'\ntoplam RAM : {gb:.1f} GB   CPU: {os.cpu_count()} cekirdek')
if gb < 20:
    print('UYARI: 05 icin yuksek RAM calisma zamani onerilir.')

LaTeX isteğe bağlı — kurulmazsa figürler matplotlib'in kendi matematik
dizgisiyle üretilir, betikler bunu kendisi algılar.

In [ ]:
!apt-get -qq update && apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super > /dev/null
print('latex kuruldu')

## 2. Veri

Veri depoyla geliyor (`data/*.dat.gz`). İndirme Colab'da çalışmaz:
`tudes.harita.gov.tr` Türkiye dışındaki DNS çözümleyicilerinden ad
çözümlemesi yapmıyor.

In [ ]:
from ortak import oku, kes, PAPER_BAS, PAPER_BIT
s = oku()
x = kes(s, PAPER_BAS, PAPER_BIT)
print(f'ayiklanmis seri  : {len(s):,} kayit  '
      f'({s.index.min().date()} -> {s.index.max().date()})')
print(f'analiz penceresi : {len(x):,} kayit  ({PAPER_BAS} -> {PAPER_BIT})')
print(f'MSL              : {x.mean():.4f} m  [istasyonun yerel datumu]')

## 3. Ayıklama  (isteğe bağlı)

`data/bozyazi_temiz.dat.gz` zaten bu adımın çıktısı. `04` ayıklamanın
doğru şeyi sildiğini sınar.

In [ ]:
!python -u 03_veri_ayikla.py
!python -u 04_ayiklama_dogrula.py

## 4. Harmonik analiz  — **ağır adım**

Çözer, yayımlanmış Bozyazı değerleriyle karşılaştırır ve çözümü
(`coef_*.pkl`), kurulan gelgiti, artığı diske yazar.

In [ ]:
!python -u 05_harmonik_analiz.py

## 5. Gelgit düzeyleri ve gelgit dışı bileşen

İkisi de `05`'in kaydettiği çözümü okur; yeniden çözüm yapılmaz.

In [ ]:
!python -u 06_gelgit_seviyeleri.py
!python -u 07_non_tidal.py

## 6. Excel kitabı

Bütün seriler, tablolar ve figürler tek dosyada.

In [ ]:
!python -u 10_excel_olustur.py

## 7. Sonuçları GitHub'a gönder

Üretilen her şey — tablolar, figürler, çözülmüş seriler, Excel kitabı —
depoya yazılır, böylece Colab oturumu kapanınca kaybolmaz.

Bir GitHub kişisel erişim jetonu gerekiyor (`repo` yetkisi yeterli).
**Jetonu hücreye yazmayın**: Colab'ın sol panelindeki anahtar simgesinden
`GITHUB_TOKEN` adıyla saklayın ve "Notebook access"i açın. Aşağıdaki hücre
onu oradan okur, böylece defter paylaşılsa bile jeton sızmaz.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('jeton alindi')
except Exception as e:
    print('Jeton okunamadi:', e)
    print("Sol paneldeki anahtar simgesinden 'GITHUB_TOKEN' ekleyip "
          "Notebook access'i acin.")

In [ ]:
%cd /content/marine_analysis/su-seviyesi
!python -u 11_sonuclari_pushla.py "Colab kosusu - analiz sonuclari"

## 8. Çıktıları indir

Sonuçları anlatımıyla, formülleriyle ve grafikleriyle görmek için
`analiz_raporu.ipynb` defterini açın.

In [ ]:
%cd /content/marine_analysis/su-seviyesi
!ls -la *.xlsx tables figures
!zip -qr /content/bozyazi_sonuclar.zip tables figures data *.xlsx
from google.colab import files
files.download('/content/bozyazi_sonuclar.zip')